# November

## Satellites

The initial objective was to evaluate whether **night-time light detection is feasible in rural settings across four satellite products**. Black Marble and SDGSAT-1 offer dedicated night-time sensors and tailored products for night-time light analysis. For Landsat 8 and EnMAP, the expectation was that, although their standard acquisitions occur during the day, previously requested custom images might exist that would overlap with the requirements of our analysis. However, no suitable nighttime imagery was available for Landsat, and retrieving EnMAP data programmatically proved too complex for convenient access.

| Satellite    | Spectral Bands               | Imaging Time (local time)  | Resolution         | Use |
| ------------ | ---------------------------- | -------------------------- | ------------------ | --- |
| Black Marble | Day/Night Band (DNB)         | 1:30 AM         | 500 m              | ✅   |
| SDGSAT-1     | Panchromatic  | 9:30 PM         | P: 10 m, RGB: 40 m | ✅   |
| Landsat 8    | RGB           | 10:00 AM        | 30 m               | 🚫  |
| EnMAP        | RGB           | 11:00 AM        | 30 m               | 🚫  |

Black Marble provides daily imagery along with monthly and yearly composite products. For SDGSAT-1, the currently available tiles are shown below:

In [1]:
import folium
import pandas as pd
from shapely import unary_union

from conflict_monitoring_ntl.case_studies import get_tiles_paths
from conflict_monitoring_ntl.utils import get_raster_gdf


tiles_paths = get_tiles_paths()

gdfs = []

for tiles_tuple in tiles_paths:
    gdfs.append(get_raster_gdf(*tiles_tuple, crs="EPSG:4326"))

gdf = pd.concat(gdfs)
global_centroid = unary_union(gdf.geometry).centroid

m = folium.Map(
    location=[global_centroid.y, global_centroid.x],
    zoom_start=3,
    tiles="CartoDB positron",
)

for _, row in gdf.iterrows():
    folium.GeoJson(
        data=row.geometry,
        name="Tiles",
        style_function=lambda _: {"fillColor": "blue", "color": "blue", "weight": 1},
    ).add_to(m)

m

Given the limited country-level coverage, we focus on administrative level 2 units from the Database of Global Administrative Areas (GADM), as illustrated by the example of South Sudan:

In [2]:
import pygadm
from conflict_monitoring_ntl.utils import get_raster_polygon, reproject_gdf
from conflict_monitoring_ntl.viz import plot_admin_map_with_tiles


country = "South Sudan"

tiles_paths = get_tiles_paths(country)
polygon, crs = get_raster_polygon(*tiles_paths)

raster_gdf = get_raster_gdf(*tiles_paths, crs="EPSG:4326")
admin_gdf = pygadm.Items(name=country, content_level=2)
country_gdf = pygadm.Items(name=country, content_level=0)

admin_gdf = reproject_gdf(admin_gdf, crs)
admin_gdf["is_within_raster"] = admin_gdf["geometry_proj"].apply(
    lambda geom: geom.within(polygon)
)

plot_admin_map_with_tiles(country_gdf, raster_gdf, admin_gdf, zoom_start=6)

## Data Analysis

### Noisy Observations

We currently have only one SDGSAT-1 tile available per location, and both SDGSAT-1 and Black Marble daily data are quite noisy. Although threshold values can be adapted for different regions or sourced from the literature, the results remain quite unstable on a daily basis.

In [ ]:
from conflict_monitoring_ntl.case_studies import get_county_ids, get_date
from conflict_monitoring_ntl.satellites import BlackMarbleEE, SDGSat
from conflict_monitoring_ntl.transform import RasterPipeline
from conflict_monitoring_ntl.utils import get_gdf_for_admin
from rasterio.enums import Resampling

from conflict_monitoring_ntl.viz import plot_tile_comparison

rasters = [SDGSat(), BlackMarbleEE()]

country = "Sudan"
date, county_id = get_date(country), get_county_ids(country)[0]
county_gdf = get_gdf_for_admin(county_id)

transformations = [{"reproject_match": {"resampling": Resampling.bilinear}}, {}]
pipeline = RasterPipeline(county_gdf, date, rasters, transformations)
ds = pipeline.run()

plot_tile_comparison(
    arr_left=ds.sdgsat_radiance,
    arr_right=ds.black_marble_radiance,
    title_left="SDGSat radiance (nW/cm²/sr)",
    title_right="Black Marble radiance (nW/cm²/sr)",
    clim_left=(7, 7.5),
    clim_right=(0, 2),
)

:Layout
   .Image.I  :Image   [lon,lat]   (sdgsat_radiance)
   .Image.II :Image   [lon,lat]   (black_marble_radiance)

### Overlap with Surface Data

Inspired in part by [Bara & Sticher](https://rdcu.be/eNT3P), this analysis leverages Global Human Settlement Layer (GHSL) population and surface data to compare against night-time light measurements.

In [5]:
from conflict_monitoring_ntl.satellites import GHSLSurface


country = "Sudan"
date, county_id = get_date(country), get_county_ids(country)[0]
county_gdf = get_gdf_for_admin(county_id)

rasters = [BlackMarbleEE(), GHSLSurface()]

transformations = [{"reproject_match": {"resampling": Resampling.bilinear}}, {}]
pipeline = RasterPipeline(county_gdf, date, rasters, transformations)
ds = pipeline.run()

2025-11-05 10:50:07,549 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-05 10:50:07,556 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-05 10:50:07,588 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-05 10:50:07,589 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-05 10:50:07,598 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-05 10:50:07,601 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.goo

In [6]:
plot_tile_comparison(
    arr_left=ds.ghsl_surface,
    arr_right=ds.black_marble_radiance,
    title_left="GHSL Surface",
    title_right="Black Marble radiance (nW/cm²/sr)",
    cmap_left="viridis",
    clim_left=(0, 50),
    clim_right=(0, 2),
)

:Layout
   .Image.I  :Image   [lon,lat]   (ghsl_surface)
   .Image.II :Image   [lon,lat]   (black_marble_radiance)

A comparison with satellite imagery reveals that daily sensor measurements — yellow for Black Marble and blue for SDGSat — do not overlap with the GHSL Surface data (red), and, intuitively, they fail to correspond with actual settlement locations. 

In [7]:
rasters = [BlackMarbleEE(), SDGSat(), GHSLSurface()]

transformations = [
    {"reproject_match": {"resampling": Resampling.bilinear}},
    {"reproject_match": {"resampling": Resampling.bilinear}},
    {},
]
pipeline = RasterPipeline(county_gdf, date, rasters, transformations)
ds = pipeline.run()

2025-11-05 10:51:53,717 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-05 10:51:53,743 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-05 10:51:53,766 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-05 10:51:53,779 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-05 10:51:53,907 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-05 10:51:53,957 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.goo

In [9]:
from conflict_monitoring_ntl.utils import binarize_xarray


binary_surface = binarize_xarray(ds.ghsl_surface, 0)
thresholded_binary = binary_surface.where(binary_surface > 0)

default_kwargs = {
    "x": "lon",
    "y": "lat",
    "crs": binary_surface.rio.crs,
    "clim": (0, 1),
    "geo": True,
    "colorbar": False,
    "height": 500,
    "width": 600,
    "alpha": 0.5,
}

ghsl = thresholded_binary.hvplot.image(
    cmap=["#FF0000"], tiles="EsriImagery", title="", **default_kwargs
)

binary_surface = binarize_xarray(ds.black_marble_radiance, 1)
thresholded_binary = binary_surface.where(binary_surface > 0)

black_marble = thresholded_binary.hvplot.image(cmap=["#EAFF00"], **default_kwargs)

binary_surface = binarize_xarray(ds.sdgsat_dn, 1.05)
thresholded_binary = binary_surface.where(binary_surface > 0)

sdg_sat = thresholded_binary.hvplot.image(cmap=["#00FFEA"], **default_kwargs)

ghsl * black_marble * sdg_sat

:Overlay
   .WMTS.I    :WMTS   [Longitude,Latitude]
   .Image.I   :Image   [lon,lat]   (ghsl_surface)
   .Image.II  :Image   [lon,lat]   (black_marble_radiance)
   .Image.III :Image   [lon,lat]   (sdgsat_dn)

### Temporal Resolution

Given the excessive noise in daily data observed from the previous tiles, relying exclusively on daily measurements may be misleading. For this reason, daily Black Marble observations are examined in parallel with monthly and yearly composites to assess data quality and variability.

As shown below, daily and monthly composites exhibit minimal overlap, and the yearly composite is nearly entirely unlit. For context, the monthly and yearly composites are generated as follows:

1. Outliers in the night-time light (NTL) data are excluded using the Boxplot method, removing observations outside the range of Q1 - 1.5×IQR to Q3 + 1.5×IQR.
2. The composites are calculated by averaging the remaining values after outlier removal.
3. To eliminate residual background noise, composite values with radiance below 0.5 nW·cm⁻²·sr⁻¹ are set to zero.

In [10]:
from conflict_monitoring_ntl.satellites import BlackMarblePy, GHSLPopulation


rasters = [
    GHSLPopulation(),
    BlackMarblePy(frequency="daily"),
    BlackMarblePy(frequency="monthly"),
    BlackMarblePy(frequency="annual"),
]

transformations = [
    {"reproject_match": {"resampling": Resampling.sum}},
    {},
    {},
    {},
]
pipeline = RasterPipeline(county_gdf, date, rasters, transformations)
ds = pipeline.run()

2025-11-05 10:55:08,826 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-05 10:55:08,873 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-05 10:55:08,874 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-05 10:55:08,892 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-05 10:55:08,943 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.googleapis.com. Connection pool size: 10
2025-11-05 10:55:08,943 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: earthengine-highvolume.goo

OBTAINING MANIFEST...:   0%|          | 0/1 [00:00<?, ?it/s]

2025-11-05 10:55:24,603 - httpx - INFO - HTTP Request: GET https://ladsweb.modaps.eosdis.nasa.gov/api/v1/files?product=VNP46A2&collection=5200&dateRanges=2023-12-12..2023-12-12&areaOfInterest=x24.96y11.45%2Cx26.09y12.97 "HTTP/1.1 200 OK"


QUEUEING TASKS | Downloading (37.6 MB)...:   0%|          | 0/1 [00:00<?, ?file/s]

PROCESSING TASKS | Downloading (37.6 MB)...:   0%|          | 0/1 [00:00<?, ?file/s]

...

2025-11-05 10:55:40,325 - httpx - INFO - HTTP Request: GET https://ladsweb.modaps.eosdis.nasa.gov/archive/allData/5200/VNP46A4/2023/001/VNP46A4.A2023001.h20v07.002.2025161155119.h5 "HTTP/1.1 200 OK"


  0%|          | 0.00/70.1M [00:00<?, ?B/s]

COLLECTING RESULTS | Downloading (70.1 MB)...:   0%|          | 0/1 [00:00<?, ?file/s]

COLLATING TILES | Processing...:   0%|          | 0/1 [00:00<?, ?date/s]

/Users/jan.kokla/Documents/EPFL/conflict-monitoring-ntl/src/conflict_monitoring_ntl/transform.py:97: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  return xr.merge(processed).rio.write_crs("EPSG:4326")


In [11]:
opts = {"labelled": [], "xticks": [], "yticks": []}

daily_ds = ds.black_marble_radiance_daily
daily = ds.black_marble_radiance_daily.hvplot.image(
    x="lon",
    y="lat",
    crs=daily_ds.rio.crs,
    clim=(0, 2),
    cmap="inferno",
    title="Black Marble - Daily",
    geo=True,
).opts(labelled=[])

monthly = ds.black_marble_radiance_monthly.hvplot.image(
    x="lon",
    y="lat",
    crs=daily_ds.rio.crs,
    clim=(0, 2),
    cmap="inferno",
    title="Black Marble - Monthly",
    geo=True,
).opts(labelled=[])

annual = ds.black_marble_radiance_annual.hvplot.image(
    x="lon",
    y="lat",
    crs=daily_ds.rio.crs,
    clim=(0, 2),
    cmap="inferno",
    title="Black Marble - Annual",
    geo=True,
).opts(labelled=[])

ghsl = ds.ghsl_population.hvplot.image(
    x="lon",
    y="lat",
    crs=daily_ds.rio.crs,
    clim=(0, 500),
    cmap="viridis",
    title="GHSL Population",
    geo=True,
).opts(labelled=[])

plot = (daily + monthly + annual + ghsl).cols(2)
plot

:Layout
   .Image.I   :Image   [lon,lat]   (black_marble_radiance_daily)
   .Image.II  :Image   [lon,lat]   (black_marble_radiance_monthly)
   .Image.III :Image   [lon,lat]   (black_marble_radiance_annual)
   .Image.IV  :Image   [lon,lat]   (ghsl_population)

### Quality Flags

Beyond temporal differences, Black Marble also flags pixels with potentially unreliable values due to various factors. The example below illustrates a composite generated using a three-day window.

In [12]:
import pandas as pd

date_range = pd.date_range(start="2023-12-02", end="2023-12-04")
ds = BlackMarblePy("daily", drop_values_by_quality_flag=[2, 255]).raster(county_gdf, date_range)

OBTAINING MANIFEST...:   0%|          | 0/1 [00:00<?, ?it/s]

2025-11-05 10:55:50,151 - httpx - INFO - HTTP Request: GET https://ladsweb.modaps.eosdis.nasa.gov/api/v1/files?product=VNP46A2&collection=5200&dateRanges=2023-12-02..2023-12-04&areaOfInterest=x24.96y11.45%2Cx26.09y12.97 "HTTP/1.1 200 OK"


QUEUEING TASKS | Downloading (110.2 MB)...:   0%|          | 0/3 [00:00<?, ?file/s]

PROCESSING TASKS | Downloading (110.2 MB)...:   0%|          | 0/3 [00:00<?, ?file/s]

COLLECTING RESULTS | Downloading (110.2 MB)...:   0%|          | 0/3 [00:00<?, ?file/s]

COLLATING TILES | Processing...:   0%|          | 0/3 [00:00<?, ?date/s]

In [13]:
default_kwargs = {
    "x": "lon",
    "y": "lat",
    "crs": ds.rio.crs,
    "clim": (0, 2),
    "cmap": "inferno",
    "geo": True,
    "colorbar": False,
    "frame_width": 250,
    "aspect": 1,
}

In [14]:
first = ds.isel(time=-1).black_marble_radiance_daily.hvplot.image(
    title="Black Marble - 2023-12-04 \n*no fill",
    **default_kwargs
)

second_ds = ds.isel(time=slice(1, 3)).ffill(dim="time")
second = second_ds.isel(time=-1).black_marble_radiance_daily.hvplot.image(
    title="Black Marble - 2023-12-04 \n*filled with T-1",
    **default_kwargs
)

third_ds = ds.ffill(dim="time")
third = third_ds.isel(time=-1).black_marble_radiance_daily.hvplot.image(
    title="Black Marble - 2023-12-04 \n*filled with T-1 + T-2",
    **default_kwargs
)

first + second + third

:Layout
   .Image.I   :Image   [lon,lat]   (black_marble_radiance_daily)
   .Image.II  :Image   [lon,lat]   (black_marble_radiance_daily)
   .Image.III :Image   [lon,lat]   (black_marble_radiance_daily)

### Urban-Rural Differences

I extended the analysis to an urban case study, using Winterthur, Switzerland, to assess whether the same challenges are equally evident in a more developed setting. As shown below, the overlap appears intuitively much stronger in the urban setting.

In [15]:
from conflict_monitoring_ntl.satellites import GHSLSurface
import datetime
import geopandas as gpd
import pygadm


date = datetime.date(2022, 3, 22)
county_gdf = pygadm.Items(name="Winterthur", content_level=2)
county_gdf = gpd.GeoDataFrame(geometry=county_gdf.geometry)
county_gdf = county_gdf.set_crs("EPSG:4326")

rasters = [SDGSat(), GHSLSurface()]

transformations = [{"reproject_match": {"resampling": Resampling.bilinear}}, {}]
pipeline = RasterPipeline(county_gdf, date, rasters, transformations)
ds = pipeline.run()

/Users/jan.kokla/Documents/EPFL/conflict-monitoring-ntl/src/conflict_monitoring_ntl/transform.py:97: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  return xr.merge(processed).rio.write_crs("EPSG:4326")


In [16]:
plot_tile_comparison(
    arr_left=ds.ghsl_surface,
    arr_right=ds.sdgsat_dn,
    title_left="GHSL Surface",
    title_right="SDGSat DN",
    cmap_left="viridis",
    clim_left=(0, 5000),
    clim_right=(0, 20),
)

:Layout
   .Image.I  :Image   [lon,lat]   (ghsl_surface)
   .Image.II :Image   [lon,lat]   (sdgsat_dn)

Additionally, the precision-recall curve behaves as expected:

In [17]:
import numpy as np
import hvplot.pandas  # noqa
from conflict_monitoring_ntl.utils import (
    binarize_xarray,
    get_combined_mask,
    get_non_nan_flat_array,
    get_precision_recall,
)

mask = get_combined_mask(ds)
ds = ds.where(mask)

ghsl_pop_binary = binarize_xarray(ds.ghsl_surface, 0)
y_true = get_non_nan_flat_array(ghsl_pop_binary)

sdgsat_thresholds = np.arange(1.01, 2.01, 0.01).tolist()
df = get_precision_recall(ds.sdgsat_dn, y_true, sdgsat_thresholds)

Computing precision/recall: 100%|██████████| 100/100 [00:01<00:00, 86.56it/s]


In [18]:
df.hvplot.scatter(
    x="recall",
    y="precision",
    c="threshold",
    height=400,
    width=500,
    clabel="threshold",
    cmap="cividis",
)

:Scatter   [recall]   (precision,threshold)

There is noticeably less variation across the different composited rasters.

In [19]:
import xarray as xr

date = datetime.date(2022, 3, 22)
gdf = pygadm.Items(name="Winterthur", content_level=2)
gdf = gpd.GeoDataFrame(geometry=gdf.geometry)
gdf = gdf.set_crs("EPSG:4326")

frequencies = ["daily", "monthly", "annual"]
rasters = [BlackMarblePy(frequency=f).raster(gdf, date) for f in frequencies]

ds = xr.merge(rasters).rio.write_crs("EPSG:4326")

OBTAINING MANIFEST...:   0%|          | 0/1 [00:00<?, ?it/s]

2025-11-05 10:56:39,799 - httpx - INFO - HTTP Request: GET https://ladsweb.modaps.eosdis.nasa.gov/api/v1/files?product=VNP46A2&collection=5200&dateRanges=2022-03-22..2022-03-22&areaOfInterest=x8.61y47.41%2Cx8.93y47.6 "HTTP/1.1 200 OK"


QUEUEING TASKS | Downloading (24.7 MB)...:   0%|          | 0/1 [00:00<?, ?file/s]

PROCESSING TASKS | Downloading (24.7 MB)...:   0%|          | 0/1 [00:00<?, ?file/s]

2025-11-05 10:56:40,785 - httpx - INFO - HTTP Request: GET https://ladsweb.modaps.eosdis.nasa.gov/archive/allData/5200/VNP46A2/2022/081/VNP46A2.A2022081.h18v04.002.2025117092422.h5 "HTTP/1.1 200 OK"


...

2025-11-05 10:56:59,147 - httpx - INFO - HTTP Request: GET https://ladsweb.modaps.eosdis.nasa.gov/archive/allData/5200/VNP46A4/2022/001/VNP46A4.A2022001.h18v04.002.2025156103302.h5 "HTTP/1.1 200 OK"


  0%|          | 0.00/123M [00:00<?, ?B/s]

COLLECTING RESULTS | Downloading (122.6 MB)...:   0%|          | 0/1 [00:00<?, ?file/s]

COLLATING TILES | Processing...:   0%|          | 0/1 [00:00<?, ?date/s]

/var/folders/xk/43532ls14wn5d_k035nfm0240000gp/T/ipykernel_20985/273046653.py:11: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds = xr.merge(rasters).rio.write_crs("EPSG:4326")


In [20]:
default_kwargs = {
    "x": "lon",
    "y": "lat",
    "crs": daily_ds.rio.crs,
    "clim": (0, 2),
    "cmap": "inferno",
    "geo": True,
    "colorbar": False,
    "frame_width": 250,
    "aspect": 1
}

daily_ds = ds.black_marble_radiance_daily
daily = ds.black_marble_radiance_daily.hvplot.image(
    title="Black Marble - Daily",
    **default_kwargs
)

monthly = ds.black_marble_radiance_monthly.hvplot.image(
    title="Black Marble - Monthly",
    **default_kwargs
)

annual = ds.black_marble_radiance_annual.hvplot.image(
    title="Black Marble - Annual",
    **default_kwargs
)

daily + monthly + annual

:Layout
   .Image.I   :Image   [lon,lat]   (black_marble_radiance_daily)
   .Image.II  :Image   [lon,lat]   (black_marble_radiance_monthly)
   .Image.III :Image   [lon,lat]   (black_marble_radiance_annual)

## Learnings

General conclusions: 

- Daily measurements are often unreliable, highlighting the importance of **temporal resolution** and **compositing** for meaningful analysis.
- **Differences between urban and rural** areas are critical—not only in night-time light intensity but also because ground truth data in rural contexts may be uncertain and can amplify inconsistencies in the results.

SDGSAT-1 Limitations for Rural Night-Time Light Analysis:

- **Temporal resolution**: Currently, for each region, only a single SDGSAT-1 raster is available. This creates two main options for comparison: use daily Black Marble data — which tends to be quite noisy — or attempt to build a composite from the limited SDGSAT-1 observations.
- **Threshold selection**: With only sporadic daily rasters, determining optimal thresholds for binarizing and comparing night-time light signals becomes more complex and less robust.
- **Data availability**: Limited SDGSAT-1 raster coverage reduces the scope of comparison to a case-study level rather than a full systematic evaluation.

## Potential Avenues

The subsequent analysis will focus exclusively on Black Marble.

### Compositing

The variability observed in daily, monthly, and yearly night-time light (NTL) values - such as in Nyala - illustrates the limitations of coarse temporal composites for detecting conflict-driven changes. Monthly or yearly aggregation reduces temporal sensitivity and masks short-term fluctuations.

A potential direction would be to explore compositing strategies that maximize valid pixel coverage while preserving the ability to capture short-term dynamics, following approaches similar to [this](https://www.sciencedirect.com/science/article/pii/S0034425722001304).

### Urban–Rural Differences

#### Settlement Diversity

Building on the Winterthur analysis, different settlements varying in size, development level, and region could be compared to assess how population and building densities relate to NTL intensity.  

#### Refugee Settlements

Alternatively, the analysis could focus exclusively on refugee settlements, which are precisely mapped by the UNHCR. This would provide a reliable ground truth for comparison with other human settlement datasets and NTL-derived estimates (inspiration from [here](https://www.mdpi.com/2072-4292/13/18/3574)).
